# 가우스 소거법과 LU 분해

> 선형대수 3강 · 선형방정식계와 행렬대수

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [가우스 소거법과 LU 분해](https://mioon1402.github.io/timeseriesdata/linalg/L03-elimination.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 식을 빼도 되는가

## 1. 소거를 눈으로 보기

## 2. 피봇과 상삼각행렬 U

## 3. 피봇이 0이면

## 4. 기본행렬 — 행 연산도 행렬 곱이다

## 5. A = LU 유도

## 6. numpy 로 직접 구현하기

**3-1. 소거를 한 줄씩 따라가기**

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)   # 보기 좋게

A = np.array([[ 2.,  1., 1.],
              [ 4., -6., 0.],
              [-2.,  7., 2.]])
b = np.array([5., -2., 9.])

U = A.copy()
c = b.copy()
L = np.eye(3)              # 배수를 적어둘 자리

for j in range(3):                     # j 번째 열을 소거
    for i in range(j + 1, 3):          # 그 아래 행들에 대해
        m = U[i, j] / U[j, j]          # 배수 = 없앨 값 ÷ 피봇
        L[i, j] = m                    # 버리지 말고 적어둔다
        U[i] -= m * U[j]
        c[i] -= m * c[j]
        print(f"{i+1}행 -= {m:.3g} × {j+1}행")
    print(U, "\n")

**3-2. L 과 U 확인, 그리고 A = LU**

In [ ]:
print("L =")
print(L)
print("\nU =")
print(U)
print("\nL @ U =")
print(L @ U)
print("\nA 와 같은가:", np.allclose(L @ U, A))

**3-3. 후진 대입으로 해 구하기**

In [ ]:
# U x = c 를 아래 행부터 푼다
n = 3
x = np.zeros(n)
for i in range(n - 1, -1, -1):
    이미_구한_부분 = U[i, i+1:] @ x[i+1:]
    x[i] = (c[i] - 이미_구한_부분) / U[i, i]
    print(f"x[{i}] = ({c[i]:.3g} - {이미_구한_부분:.3g}) / {U[i,i]:.3g} = {x[i]:.3g}")

print("\n직접 구한 해 :", x)
print("numpy 의 해  :", np.linalg.solve(A, b))

**3-4. 피봇이 0이면 무슨 일이**

In [ ]:
나쁜A = np.array([[0., 1.],
                 [2., 3.]])
print("첫 피봇 =", 나쁜A[0, 0], " → 0 이라 나눌 수 없다")
print()

# 해결책: 행을 바꾼다
바꾼A = 나쁜A[[1, 0]]          # 1행과 0행을 교환
print("행 교환 후:")
print(바꾼A)
print("이제 첫 피봇 =", 바꾼A[0, 0], " → 진행 가능")
print()

# 아래가 전부 0이면 교환해도 소용없다 = 특이행렬
특이 = np.array([[1., 2.],
                 [2., 4.]])
print("특이행렬 det =", np.linalg.det(특이))
print("→ 피봇을 2개 확보할 수 없다. 역행렬 없음")

**3-5. 라이브러리와 비교 — 왜 다른가**

In [ ]:
from scipy.linalg import lu

P, L2, U2 = lu(A)

print("P (행 교환 행렬) =")
print(P)
print("\nP 가 단위행렬인가:", np.allclose(P, np.eye(3)), " ← False 면 행을 바꿨다는 뜻")
print()
print("우리가 구한 U =\n", U)
print("\nscipy 의 U   =\n", U2)
print()
print("둘 다 맞는 분해인가:")
print("  우리 것 : L @ U   == A     →", np.allclose(L @ U, A))
print("  scipy   : P @ L2 @ U2 == A →", np.allclose(P @ L2 @ U2, A))

**3-6. 분해해두면 정말 빠른가**

In [ ]:
import time
from scipy.linalg import lu_factor, lu_solve

rng = np.random.default_rng(0)
n = 300
큰A = rng.random((n, n)) + n * np.eye(n)     # 잘 풀리는 행렬로
b들 = rng.random((50, n))                    # 우변 50개

t = time.perf_counter()
for bb in b들:
    np.linalg.solve(큰A, bb)                 # 매번 처음부터
t1 = time.perf_counter() - t

t = time.perf_counter()
lu_piv = lu_factor(큰A)                      # 한 번만 분해
for bb in b들:
    lu_solve(lu_piv, bb)                     # 대입 두 번씩
t2 = time.perf_counter() - t

print(f"매번 소거      : {t1*1000:7.1f} ms")
print(f"한 번 분해 후  : {t2*1000:7.1f} ms")
print(f"→ 약 {t1/t2:.1f} 배 빠름  (n={n}, 우변 {len(b들)}개)")

**3-7. 연습문제**

In [ ]:
# 문제 1. 아래 행렬을 손으로 소거해 U 와 L 을 구하고, 코드로 확인해보세요.
#         [[1, 2],
#          [3, 8]]

# 문제 2. 그 A 에 대해 b = [5, 19] 일 때 해를 후진 대입으로 구해보세요.

# 문제 3. [[0, 1], [1, 0]] 은 피봇이 0 입니다.
#         행을 바꾸면 무엇이 되나요? det 는 얼마인가요?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1 — 2행에서 1행의 3배를 뺀다
A1 = np.array([[1., 2.], [3., 8.]])
m = A1[1, 0] / A1[0, 0]           # = 3
U1 = A1.copy(); U1[1] -= m * U1[0]
L1 = np.array([[1., 0.], [m, 1.]])
print("배수 =", m)
print("U =\n", U1)
print("L =\n", L1)
print("L@U == A :", np.allclose(L1 @ U1, A1))

# 문제 2
b1 = np.array([5., 19.])
c1 = b1.copy(); c1[1] -= m * c1[0]        # 우변도 같이 소거
y = c1[1] / U1[1, 1]
x = (c1[0] - U1[0, 1] * y) / U1[0, 0]
print("\n해 =", [x, y], " 확인:", np.linalg.solve(A1, b1))

# 문제 3 — 행을 바꾸면 단위행렬이 된다
A3 = np.array([[0., 1.], [1., 0.]])
print("\n행 교환 후:\n", A3[[1, 0]])
print("det =", np.linalg.det(A3), " ← 행을 한 번 바꾸면 부호가 뒤집힌다 (12강)")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)